## Import config

In [ ]:
# Корень текущей ветки: config/srcc этой же ветки, без зависимости от cwd.
from pathlib import Path
import sys, os
_cwd = Path.cwd().resolve()
_expected = 'dynamic'
_candidates = [_cwd, _cwd / _expected, _cwd.parent / _expected]
PROJECT_ROOT = next((x for x in _candidates if (x / 'srcc' / 'nbc_project').is_dir() and x.name == _expected), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Запускайте ноутбук из папки {_expected} или её родителя')
os.chdir(PROJECT_ROOT)
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
SRC_PATH

In [ ]:
import pandas as pd
import numpy as np
from math import ceil

from rail_connectors import connect
import rail_connectors

# Списки фичей вынесены в configs/feature_lists.json
from nbc_project.feature_lists import load_feature_lists, get_feature_list, select_existing_columns

FEATURE_LISTS = load_feature_lists()

postfix = "dec_active_base_dynamic_2026"

TARGET_MONTH = '2026-12-01'

# Редактируй этот блок перед запуском нового месяца.
BASE_VARIANT = "active_base"
MODEL_VARIANT = "dynamic"

# В декабре читаем октябрьские транзакции (лаг -2), а не сентябрьский benchmark.
TRANSACTION_SOURCE = 'sandbox_crft.cb_charges_compact_nbc_monthly_202610'
assert TARGET_MONTH == '2026-12-01'


In [ ]:
conn = connect(        
        to='IMPALA'
    )

## Витрина клиентов для скоринга

In [ ]:
# Можно заменить на свою витрину список cdi_id для скоринга
BASE_GROUPS = "sandbox_crft.pia_base_groups"
CLIENTS_FOR_LOYALTY = f'sandbox_crft.mae_client_for_loyalty_{postfix}'
# CLIENTS_FOR_LOYALTY = f'sandbox_crft.pia_client_for_loyalty_inference'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {CLIENTS_FOR_LOYALTY}''')

In [ ]:
# with conn:
#     conn.execute(f'''
#     CREATE TABLE {CLIENTS_FOR_LOYALTY}
#     TBLPROPERTIES('transactional' = 'false')
#     AS with a as (
#         select
#             customer_id,
#             abs(sum(charge_rub_amt))/count(distinct TRUNC(tt.trans_dt, 'mm')) as charge_rub_amt
#         from dm_dcp_fl.tfct_transaction tt
#         left join dm_dcp_fl.dim_mcc mcc
#             on tt.mcc_id = mcc.mcc_id 
#         where tt.mcc_id != -99
#             and customer_id is not null
#             and trans_dt >= '{start_date}' and trans_dt < '{end_date}'
#         group by customer_id
#         ),
#         b as (
#             select 
#                 a.customer_id, c.cdi_party_id, count(*)
#             from a
#             left join dm_dcp_fl.ref_customer c
#                 on c.customer_id = a.customer_id
#             where cdi_party_id is not null
#             group by a.customer_id, c.cdi_party_id
#         )
#         select distinct cast('{TARGET_MONTH}' as date) as period_start_date, cdi_party_id as cdi_id
#         from b
#         limit 200
#     ''')

In [ ]:
with conn:
    conn.execute(f'''
    CREATE TABLE {CLIENTS_FOR_LOYALTY}
    TBLPROPERTIES('transactional' = 'false')
    AS
        select
            cast('{TARGET_MONTH}' as date) as period_start_date
            , cast(cdi_id as bigint) as cdi_id
        from {BASE_GROUPS}
        where pilot_group = 'active_base'
    ''')

In [ ]:
with conn:
    d = conn.fetch(f'''select count(*), count(distinct(cdi_id)) from {CLIENTS_FOR_LOYALTY}''')
    df_all = conn.fetch(f'''select * from {CLIENTS_FOR_LOYALTY} limit 5''')
d

In [ ]:
df_all

## Витрина сценариев для клиентов

In [ ]:
SEND_TO_DBO = f"sandbox_crft.send_to_dbo_{postfix}"

# SOURCE_TABLE = "sandbox_crft.mae_nbc_optimizer_scores_fm_bin_june_cat_id"
# SOURCE_TABLE = "sandbox_crft.mae_nbc_optimizer_scores_fm_bin_august_2026_catid"

# Внешний источник уже содержит готовые 10 строк на сценарий:
# cdi_id, scenario_id, coefficient, category_id, category_percent
# EXTERNAL_SOURCE_TABLE = "sandbox_crft.external_client_scenarios"

# Пул категорий для random:
# category_id
CATEGORY_POOL_TABLE = "sandbox_crft.mae_category_pool"

# COST_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_cost_scenarios"
# UPLIFT_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_uplift_scenarios"
EXTERNAL_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_external_scenarios_202612"
RANDOM_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_random_scenarios"

SOURCE_CLIENT_COL = "cdi_id"
SOURCE_CATEGORY_COL = "category_id"

# ALL_PURCHASES_CATEGORY_ID = "d40cd3f8-5b8c-426f-84a4-46c848f7b901"
ALL_PURCHASES_CATEGORY_ID = "9a038f75-65c1-43b5-95bc-62967a8d1100"

CAPPED_CATEGORIES = (
    "9a038f75-65c1-43b5-95bc-62967a8d1133",
    "9a038f75-65c1-43b5-95bc-62967a8d1134",
)

CAPPED_CATEGORIES_SQL = ", ".join(f"'{x}'" for x in CAPPED_CATEGORIES)
# Имя декабрьского внешнего источника выведено по ноябрьскому шаблону; наличие строк проверяем ДО CREATE/DROP.


### build_random_topk_scenario_table

#### Новая

In [ ]:
def build_random_topk_scenario_table(
    conn,
    output_table: str,
    tmp_prefix: str,
    top_k_random_scenarios: int = 3,
):
    random_candidates = f"{tmp_prefix}_candidates"
    random_ranked = f"{tmp_prefix}_ranked"
    random_categories = f"{tmp_prefix}_categories"

    with conn:

        scenario_nums_sql = "\nUNION ALL\n".join(
            f"SELECT CAST({i} AS INT) AS scenario_num"
            for i in range(1, top_k_random_scenarios + 1)
        )


        # Шаг 1. Создаем random-кандидатов отдельно для каждого scenario_num
        print(f"[random] Удаляю витрину: {random_candidates}")
        conn.execute(f"DROP TABLE IF EXISTS {random_candidates}")
        print(f"[random] Витрина удалена: {random_candidates}")

        print(f"[random] Создаю витрину: {random_candidates}")
        conn.execute(f"""
            CREATE TABLE {random_candidates}
            TBLPROPERTIES('transactional' = 'false')
            AS

            WITH scenario_nums AS (
                {scenario_nums_sql}
            )

            SELECT
                CAST(b.cdi_id AS STRING) AS cdi_id,
                CAST(b.period_start_date AS DATE) AS period_start_date,
                s.scenario_num,
                CAST(cat.category_id AS STRING) AS category_id,
                RAND() AS random_score
            FROM {CLIENTS_FOR_LOYALTY} b
            CROSS JOIN scenario_nums s
            CROSS JOIN {CATEGORY_POOL_TABLE} cat
            WHERE CAST(cat.category_id AS STRING) != '{ALL_PURCHASES_CATEGORY_ID}'
        """)
        print(f"[random] Витрина создана: {random_candidates}")




        # Шаг 2. Ранжируем random-кандидатов внутри каждого scenario_num
        print(f"[random] Удаляю витрину: {random_ranked}")
        conn.execute(f"DROP TABLE IF EXISTS {random_ranked}")
        print(f"[random] Витрина удалена: {random_ranked}")

        print(f"[random] Создаю витрину: {random_ranked}")
        conn.execute(f"""
            CREATE TABLE {random_ranked}
            TBLPROPERTIES('transactional' = 'false')
            AS
            SELECT
                cdi_id,
                period_start_date,
                scenario_num,
                category_id,
                ROW_NUMBER() OVER (
                    PARTITION BY cdi_id, period_start_date, scenario_num
                    ORDER BY random_score
                ) AS random_rank
            FROM {random_candidates}
        """)
        print(f"[random] Витрина создана: {random_ranked}")


        # Шаг 3. Создаем random-категории для сценариев
        print(f"[random] Удаляю витрину: {random_categories}")
        conn.execute(f"DROP TABLE IF EXISTS {random_categories}")
        print(f"[random] Витрина удалена: {random_categories}")

        print(f"[random] Создаю витрину: {random_categories}")
        conn.execute(f"""
            CREATE TABLE {random_categories}
            TBLPROPERTIES('transactional' = 'false')
            AS
            SELECT
                cdi_id,
                period_start_date,
                CONCAT('client_random_topn_', CAST(scenario_num AS STRING)) AS offer_scenario_id,
                'random' AS offer_source,
                CAST(random_rank + 1 AS INT) AS category_rank,
                category_id,
                CASE
                    WHEN category_id IN ({CAPPED_CATEGORIES_SQL}) THEN 5
                    WHEN random_rank <= 8 THEN 5
                    ELSE
                        CASE FLOOR(RAND() * 6)
                            WHEN 0 THEN 5
                            WHEN 1 THEN 7
                            WHEN 2 THEN 9
                            WHEN 3 THEN 11
                            WHEN 4 THEN 13
                            ELSE 15
                        END
                END AS category_percent,
                CASE
                    WHEN random_rank <= 8 THEN 'fixed_5'
                    ELSE 'variable_percent'
                END AS category_role,
                CAST(NULL AS DOUBLE) AS source_score,
                CAST(1 AS INT) AS is_random_fallback,
                scenario_num
            FROM {random_ranked}
            WHERE random_rank <= 9
        """)
        print(f"[random] Витрина создана: {random_categories}")



        # Шаг 4. Собираем итоговую random-витрину
        print(f"[random] Удаляю итоговую витрину: {output_table}")
        conn.execute(f"DROP TABLE IF EXISTS {output_table}")
        print(f"[random] Итоговая витрина удалена: {output_table}")

        print(f"[random] Создаю итоговую витрину: {output_table}")
        conn.execute(f"""
            CREATE TABLE {output_table}
            TBLPROPERTIES('transactional' = 'false')
            AS

            WITH scenario_clients AS (
                SELECT
                    cdi_id,
                    period_start_date,
                    offer_scenario_id,
                    offer_source,
                    scenario_num
                FROM {random_categories}
                GROUP BY
                    cdi_id,
                    period_start_date,
                    offer_scenario_id,
                    offer_source,
                    scenario_num
                HAVING COUNT(*) = 9
                   AND COUNT(DISTINCT category_id) = 9
            ),

            mandatory_all_purchases AS (
                SELECT
                    cdi_id,
                    period_start_date,
                    offer_scenario_id,
                    offer_source,
                    CAST(1 AS INT) AS category_rank,
                    CAST('{ALL_PURCHASES_CATEGORY_ID}' AS STRING) AS category_id,
                    CAST(1 AS INT) AS category_percent,
                    'mandatory_all_purchases' AS category_role,
                    CAST(NULL AS DOUBLE) AS source_score,
                    CAST(1 AS INT) AS is_random_fallback,
                    scenario_num
                FROM scenario_clients
            ),

            final_raw AS (
                SELECT
                    cdi_id,
                    period_start_date,
                    offer_scenario_id,
                    offer_source,
                    category_rank,
                    category_id,
                    category_percent,
                    category_role,
                    source_score,
                    is_random_fallback,
                    scenario_num
                FROM mandatory_all_purchases

                UNION ALL

                SELECT
                    cdi_id,
                    period_start_date,
                    offer_scenario_id,
                    offer_source,
                    category_rank,
                    category_id,
                    category_percent,
                    category_role,
                    source_score,
                    is_random_fallback,
                    scenario_num
                FROM {random_categories}
            ),

            good_scenarios AS (
                SELECT
                    cdi_id,
                    period_start_date,
                    offer_scenario_id
                FROM final_raw
                GROUP BY
                    cdi_id,
                    period_start_date,
                    offer_scenario_id
                HAVING COUNT(*) = 10
                   AND COUNT(DISTINCT category_id) = 10
            )

            SELECT
                f.cdi_id,
                f.period_start_date,
                f.offer_scenario_id,
                f.offer_source,
                f.scenario_num,
                f.category_rank,
                f.category_id,
                CASE
                    WHEN f.category_id IN ({CAPPED_CATEGORIES_SQL})
                     AND f.category_percent > 5 THEN 5
                    ELSE f.category_percent
                END AS category_percent,
                f.category_role,
                f.source_score,
                f.is_random_fallback,
                CURRENT_TIMESTAMP() AS created_at
            FROM final_raw f
            JOIN good_scenarios g
                ON f.cdi_id = g.cdi_id
               AND f.period_start_date = g.period_start_date
               AND f.offer_scenario_id = g.offer_scenario_id
        """)
        print(f"[random] Итоговая витрина создана: {output_table}")
        print("[random] Готово")

#### Новая с непересечением категорий про путешествия

In [ ]:
# def build_random_topk_scenario_table(
#     conn,
#     output_table: str,
#     tmp_prefix: str,
#     top_k_random_scenarios: int = 3,
#     exclusive_category_1: str = None,
#     exclusive_category_2: str = None,
# ):
#     """
#     Генерация random-сценариев.

#     На каждый сценарий:
#         - 1 обязательная категория "Все покупки" = 1%
#         - 8 случайных категорий = 5%
#         - 1 случайная категория = 5/7/9/11/13/15%
#         - capped-категории всегда <= 5%
#         - exclusive_category_1 и exclusive_category_2
#           не могут находиться вместе в одном сценарии

#     Одна категория может встречаться в разных сценариях,
#     но внутри одного сценария категории уникальны.
#     """

#     if top_k_random_scenarios < 1:
#         raise ValueError("top_k_random_scenarios должен быть >= 1")

#     random_candidates = f"{tmp_prefix}_candidates"
#     random_ranked = f"{tmp_prefix}_ranked"
#     random_categories = f"{tmp_prefix}_categories"

#     # ------------------------------------------------------------------
#     # Генерируем scenario_num = 1 ... top_k_random_scenarios
#     # ------------------------------------------------------------------

#     scenario_nums_sql = "\nUNION ALL\n".join(
#         f"SELECT CAST({i} AS INT) AS scenario_num"
#         for i in range(1, top_k_random_scenarios + 1)
#     )

#     # ------------------------------------------------------------------
#     # Логика взаимоисключающих категорий
#     # ------------------------------------------------------------------

#     has_exclusive_pair = (
#         exclusive_category_1 is not None
#         and exclusive_category_2 is not None
#     )

#     if has_exclusive_pair:

#         exclusive_filter_cte = f"""
#         ,
#         exclusive_filtered AS (
#             SELECT
#                 cdi_id,
#                 period_start_date,
#                 scenario_num,
#                 category_id,
#                 random_rank,

#                 MIN(
#                     CASE
#                         WHEN category_id IN (
#                             '{exclusive_category_1}',
#                             '{exclusive_category_2}'
#                         )
#                         THEN random_rank
#                     END
#                 ) OVER (
#                     PARTITION BY
#                         cdi_id,
#                         period_start_date,
#                         scenario_num
#                 ) AS first_exclusive_rank

#             FROM {random_ranked}
#         ),

#         eligible_categories AS (
#             SELECT
#                 cdi_id,
#                 period_start_date,
#                 scenario_num,
#                 category_id,
#                 random_rank

#             FROM exclusive_filtered

#             WHERE
#                 category_id NOT IN (
#                     '{exclusive_category_1}',
#                     '{exclusive_category_2}'
#                 )

#                 OR random_rank = first_exclusive_rank
#         )
#         """

#         categories_source = "eligible_categories"

#         exclusive_having = f"""
#             AND SUM(
#                 CASE
#                     WHEN category_id IN (
#                         '{exclusive_category_1}',
#                         '{exclusive_category_2}'
#                     )
#                     THEN 1
#                     ELSE 0
#                 END
#             ) <= 1
#         """

#     else:

#         exclusive_filter_cte = f"""
#         ,
#         eligible_categories AS (
#             SELECT
#                 cdi_id,
#                 period_start_date,
#                 scenario_num,
#                 category_id,
#                 random_rank
#             FROM {random_ranked}
#         )
#         """

#         categories_source = "eligible_categories"
#         exclusive_having = ""

#     with conn:

#         # ==============================================================
#         # ШАГ 1
#         # Для каждого клиента и каждого scenario_num
#         # создаём полный случайный пул категорий
#         # ==============================================================

#         print(f"[random] Удаляю витрину: {random_candidates}")
#         conn.execute(
#             f"DROP TABLE IF EXISTS {random_candidates}"
#         )

#         print(f"[random] Создаю витрину: {random_candidates}")

#         conn.execute(f"""
#             CREATE TABLE {random_candidates}
#             TBLPROPERTIES('transactional' = 'false')
#             AS

#             WITH scenario_nums AS (
#                 {scenario_nums_sql}
#             ),

#             category_pool AS (
#                 SELECT DISTINCT
#                     CAST(category_id AS STRING) AS category_id
#                 FROM {CATEGORY_POOL_TABLE}
#                 WHERE CAST(category_id AS STRING)
#                       != '{ALL_PURCHASES_CATEGORY_ID}'
#             )

#             SELECT
#                 CAST(b.cdi_id AS STRING) AS cdi_id,
#                 CAST(b.period_start_date AS DATE)
#                     AS period_start_date,

#                 s.scenario_num,

#                 cat.category_id,

#                 RAND() AS random_score

#             FROM {CLIENTS_FOR_LOYALTY} b

#             CROSS JOIN scenario_nums s
#             CROSS JOIN category_pool cat
#         """)

#         print(f"[random] Витрина создана: {random_candidates}")

#         # ==============================================================
#         # ШАГ 2
#         # Независимый random ranking категорий
#         # внутри каждого сценария
#         # ==============================================================

#         print(f"[random] Удаляю витрину: {random_ranked}")
#         conn.execute(
#             f"DROP TABLE IF EXISTS {random_ranked}"
#         )

#         print(f"[random] Создаю витрину: {random_ranked}")

#         conn.execute(f"""
#             CREATE TABLE {random_ranked}
#             TBLPROPERTIES('transactional' = 'false')
#             AS

#             SELECT
#                 cdi_id,
#                 period_start_date,
#                 scenario_num,
#                 category_id,

#                 ROW_NUMBER() OVER (
#                     PARTITION BY
#                         cdi_id,
#                         period_start_date,
#                         scenario_num
#                     ORDER BY random_score
#                 ) AS random_rank

#             FROM {random_candidates}
#         """)

#         print(f"[random] Витрина создана: {random_ranked}")

#         # ==============================================================
#         # ШАГ 3
#         # Убираем конфликт между двумя exclusive-категориями.
#         #
#         # Если выпали обе:
#         #
#         # rank 2 → category_A
#         # rank 5 → category_B
#         #
#         # оставляем category_A,
#         # category_B исключаем.
#         #
#         # После этого заново ранжируем список и берём первые 9.
#         # ==============================================================

#         print(f"[random] Удаляю витрину: {random_categories}")

#         conn.execute(
#             f"DROP TABLE IF EXISTS {random_categories}"
#         )

#         print(f"[random] Создаю витрину: {random_categories}")

#         conn.execute(f"""
#             CREATE TABLE {random_categories}
#             TBLPROPERTIES('transactional' = 'false')
#             AS

#             WITH base AS (
#                 SELECT *
#                 FROM {random_ranked}
#             )

#             {exclusive_filter_cte}

#             ,

#             reranked AS (

#                 SELECT
#                     cdi_id,
#                     period_start_date,
#                     scenario_num,
#                     category_id,

#                     ROW_NUMBER() OVER (
#                         PARTITION BY
#                             cdi_id,
#                             period_start_date,
#                             scenario_num

#                         ORDER BY random_rank
#                     ) AS scenario_rank

#                 FROM {categories_source}
#             )

#             SELECT
#                 cdi_id,

#                 period_start_date,

#                 CONCAT(
#                     'client_random_topn_',
#                     CAST(scenario_num AS STRING)
#                 ) AS offer_scenario_id,

#                 'random' AS offer_source,

#                 -- rank 1 будет занят "Все покупки"
#                 CAST(scenario_rank + 1 AS INT)
#                     AS category_rank,

#                 category_id,

#                 CASE

#                     -- capped категории всегда <= 5%
#                     WHEN category_id IN (
#                         {CAPPED_CATEGORIES_SQL}
#                     )
#                     THEN 5

#                     -- первые 8 random-категорий
#                     WHEN scenario_rank <= 8
#                     THEN 5

#                     -- последняя, девятая категория
#                     ELSE
#                         CASE FLOOR(RAND() * 6)
#                             WHEN 0 THEN 5
#                             WHEN 1 THEN 7
#                             WHEN 2 THEN 9
#                             WHEN 3 THEN 11
#                             WHEN 4 THEN 13
#                             ELSE 15
#                         END

#                 END AS category_percent,

#                 CASE
#                     WHEN scenario_rank <= 8
#                     THEN 'fixed_5'
#                     ELSE 'variable_percent'
#                 END AS category_role,

#                 CAST(NULL AS DOUBLE)
#                     AS source_score,

#                 CAST(1 AS INT)
#                     AS is_random_fallback,

#                 scenario_num

#             FROM reranked

#             WHERE scenario_rank <= 9
#         """)

#         print(f"[random] Витрина создана: {random_categories}")

#         # ==============================================================
#         # ШАГ 4
#         # Финальная проверка + добавление "Все покупки"
#         # ==============================================================

#         print(
#             f"[random] Удаляю итоговую витрину: "
#             f"{output_table}"
#         )

#         conn.execute(
#             f"DROP TABLE IF EXISTS {output_table}"
#         )

#         print(
#             f"[random] Создаю итоговую витрину: "
#             f"{output_table}"
#         )

#         conn.execute(f"""
#             CREATE TABLE {output_table}
#             TBLPROPERTIES('transactional' = 'false')
#             AS

#             WITH scenario_clients AS (

#                 SELECT
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id,
#                     offer_source,
#                     scenario_num

#                 FROM {random_categories}

#                 GROUP BY
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id,
#                     offer_source,
#                     scenario_num

#                 HAVING
#                     COUNT(*) = 9

#                     AND COUNT(DISTINCT category_id) = 9

#                     {exclusive_having}
#             ),

#             mandatory_all_purchases AS (

#                 SELECT
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id,
#                     offer_source,

#                     CAST(1 AS INT)
#                         AS category_rank,

#                     CAST(
#                         '{ALL_PURCHASES_CATEGORY_ID}'
#                         AS STRING
#                     ) AS category_id,

#                     CAST(1 AS INT)
#                         AS category_percent,

#                     'mandatory_all_purchases'
#                         AS category_role,

#                     CAST(NULL AS DOUBLE)
#                         AS source_score,

#                     CAST(1 AS INT)
#                         AS is_random_fallback,

#                     scenario_num

#                 FROM scenario_clients
#             ),

#             selected_random_categories AS (

#                 SELECT
#                     r.cdi_id,
#                     r.period_start_date,
#                     r.offer_scenario_id,
#                     r.offer_source,
#                     r.category_rank,
#                     r.category_id,
#                     r.category_percent,
#                     r.category_role,
#                     r.source_score,
#                     r.is_random_fallback,
#                     r.scenario_num

#                 FROM {random_categories} r

#                 JOIN scenario_clients s

#                     ON r.cdi_id = s.cdi_id

#                    AND r.period_start_date
#                        = s.period_start_date

#                    AND r.offer_scenario_id
#                        = s.offer_scenario_id
#             ),

#             final_raw AS (

#                 SELECT
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id,
#                     offer_source,
#                     category_rank,
#                     category_id,
#                     category_percent,
#                     category_role,
#                     source_score,
#                     is_random_fallback,
#                     scenario_num

#                 FROM mandatory_all_purchases

#                 UNION ALL

#                 SELECT
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id,
#                     offer_source,
#                     category_rank,
#                     category_id,
#                     category_percent,
#                     category_role,
#                     source_score,
#                     is_random_fallback,
#                     scenario_num

#                 FROM selected_random_categories
#             ),

#             good_scenarios AS (

#                 SELECT
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id

#                 FROM final_raw

#                 GROUP BY
#                     cdi_id,
#                     period_start_date,
#                     offer_scenario_id

#                 HAVING
#                     COUNT(*) = 10

#                     AND COUNT(DISTINCT category_id) = 10
#             )

#             SELECT
#                 f.cdi_id,

#                 f.period_start_date,

#                 f.offer_scenario_id,

#                 f.offer_source,

#                 f.scenario_num,

#                 f.category_rank,

#                 f.category_id,

#                 CASE
#                     WHEN f.category_id IN (
#                         {CAPPED_CATEGORIES_SQL}
#                     )
#                      AND f.category_percent > 5

#                     THEN 5

#                     ELSE f.category_percent

#                 END AS category_percent,

#                 f.category_role,

#                 f.source_score,

#                 f.is_random_fallback,

#                 CURRENT_TIMESTAMP()
#                     AS created_at

#             FROM final_raw f

#             JOIN good_scenarios g

#                 ON f.cdi_id = g.cdi_id

#                AND f.period_start_date
#                    = g.period_start_date

#                AND f.offer_scenario_id
#                    = g.offer_scenario_id
#         """)

#         print(
#             f"[random] Итоговая витрина создана: "
#             f"{output_table}"
#         )

#         print("[random] Готово")

### Вызов функций

In [ ]:
# Dynamic получает сценарии (fixed + random) из Current; НЕ запускаем случайную генерацию заново.


In [ ]:
# Dynamic читает те же сценарии, что Current; никакой дополнительной рандомизации.
SHARED_SCENARIOS = 'sandbox_crft.send_to_dbo_dec_active_base_current_2026'
with conn:
    _n = conn.fetch(f"SELECT COUNT(*) AS n FROM {SHARED_SCENARIOS} WHERE CAST(period_start_date AS DATE) = '{TARGET_MONTH}'")
    assert int(_n.iloc[0,0]) > 0, f'Нет декабрьского CURRENT источника {SHARED_SCENARIOS}; сначала current/active'
    conn.execute(f'DROP TABLE IF EXISTS {SEND_TO_DBO}')
    conn.execute(f"CREATE TABLE {SEND_TO_DBO} STORED AS PARQUET AS SELECT * FROM {SHARED_SCENARIOS} WHERE CAST(period_start_date AS DATE) = '{TARGET_MONTH}'")
print('Dynamic scenario rows reused:',int(_n.iloc[0,0]))


In [ ]:
1

## Сборка фичей

In [ ]:
cb_base = f"sandbox_crft.cb_base_client_month_{postfix}"
cb_base_scen = f"sandbox_crft.cb_base_scenario_client_month_{postfix}"
# cb_charges_compact = f"sandbox_crft.cb_charges_compact_04_06_cashback_total_pred"
# cb_charges_compact = f"sandbox_crft.cb_charges_compact_june_scenarios_train_pred"
# cb_charges_compact = f"sandbox_crft.cb_charges_compact_august_oot_train_pred"
cb_charges_compact = f"sandbox_crft.cb_charges_compact_nbc_monthly_202610"
cb_category_spend_wide = f"sandbox_crft.cb_category_spend_wide_{postfix}"
cb_offered_percent_wide = f"sandbox_crft.cb_offered_percent_wide_{postfix}"
cb_selected_history_wide = f"sandbox_crft.cb_selected_history_wide_{postfix}"
cb_acrm_monthly = f"sandbox_crft.cb_acrm_monthly_{postfix}"
cb_acrm_targets = f"sandbox_crft.cb_acrm_targets_{postfix}"
cb_acrm_lags = f"sandbox_crft.cb_acrm_lags_{postfix}"
cb_activity_features = f"sandbox_crft.cb_activity_features_{postfix}"
cb_potential_features = f"sandbox_crft.cb_potential_features_{postfix}"
cb_share_features = f"sandbox_crft.cb_share_features_{postfix}"
cb_dataset = f"sandbox_crft.cb_client_month_dataset_{postfix}"

LOYALTY_CLIENT_CATEGORIES = 'sandbox_crft.mae_loyalty_client_categories'
send_to_dbo = SEND_TO_DBO
# send_to_dbo = 'sandbox_crft.pia_final_loyal_202608'

def run_sql(conn, queries, compute_stats_table=None, preview_table=None, limit=5):
    with conn:
        for query in queries:
            conn.execute(query)

        if compute_stats_table is not None:
            conn.execute(f"compute stats {compute_stats_table}")

        if preview_table is not None:
            return conn.fetch(f"""
                select *
                from {preview_table}
                limit {limit}
            """)


def category_suffix(category_id: str) -> str:
    return str(category_id).replace("-", "")[-4:]


start_month = '2026-03-01'
end_month = '2026-12-01'
assert cb_charges_compact == TRANSACTION_SOURCE

# Общий октябрьский источник для декабрьской истории: проверяем доступные месяцы.
with conn:
    _tx_months = conn.fetch(f"SELECT TRUNC(CAST(month_date AS DATE), 'MM') AS m, COUNT(*) AS n FROM {cb_charges_compact} WHERE CAST(month_date AS DATE) >= '2026-05-01' AND CAST(month_date AS DATE) < '2026-11-01' GROUP BY TRUNC(CAST(month_date AS DATE), 'MM') ORDER BY m")
_expected_tx = {'2026-05','2026-06','2026-07','2026-08','2026-09','2026-10'}
_actual_tx = set(pd.to_datetime(_tx_months['m']).dt.strftime('%Y-%m')) if not _tx_months.empty else set()
assert _expected_tx <= _actual_tx, f'Не хватает месяцев в cb_charges_compact: {_expected_tx-_actual_tx}'
print('Месяцы истории:', sorted(_actual_tx))
print('ВНИМАНИЕ: сверить покрытие клиентов декабрьской базы этой витриной; она создана на октябрьской обучающей базе!')


In [ ]:
send_to_dbo

### 1. Базовая таблица клиент-месяц-сценарий

In [ ]:
%%time
drop = f"""
drop table if exists {cb_base}
"""

create = f"""
create table {cb_base} stored as parquet as
select distinct
    cast(cdi_id as bigint) as cdi_id,
    cast(period_start_date as timestamp) as forecast_month
from {send_to_dbo}
where period_start_date >= cast('{start_month}' as timestamp)
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)
        
    df = conn.fetch(f"""select * from {cb_base} limit 5""")
df

In [ ]:
# Диагностика покрытия декабрьской аудитории октябрьской обучающей транзакционной витриной.
# Отсутствие транзакций не равно отсутствию клиента: сравнивайте с нормальной долей активных клиентов.
with conn:
    _cov = conn.fetch(f"""
        SELECT COUNT(DISTINCT b.cdi_id) AS scoring_clients,
               COUNT(DISTINCT t.cdi_id) AS clients_with_history
        FROM {cb_base} b
        LEFT JOIN {cb_charges_compact} t
          ON b.cdi_id = t.cdi_id
         AND CAST(t.month_date AS DATE) >= '2026-05-01'
         AND CAST(t.month_date AS DATE) < '2026-11-01'
    """)
print('Сравните покрытие транзакционной историей с ожидаемой долей активных клиентов:', _cov)


In [ ]:
%%time
drop = f"""
drop table if exists {cb_base_scen}
"""

create = f"""
create table {cb_base_scen} stored as parquet as
select distinct
    cast(cdi_id as bigint) as cdi_id,
    cast(period_start_date as timestamp) as forecast_month,
    offer_scenario_id
from {send_to_dbo}
where period_start_date >= cast('{start_month}' as timestamp)
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)
        
    df = conn.fetch(f"""select * from {cb_base_scen} limit 5""")
df

### 2. Вытаскиваем уникальные категории

In [ ]:
%%time
with conn:
    categories_df = conn.fetch(f"""
        select distinct
            title_id,
            title
        from {cb_charges_compact}
        where title_id is not null
        order by title
    """)

categories_df["suffix"] = categories_df["title_id"].apply(category_suffix)

duplicates = (
    categories_df
    .groupby("suffix")
    .size()
    .reset_index(name="cnt")
    .query("cnt > 1")
)

duplicates

### 3. Общие траты + wide-траты в категориях

In [ ]:
%%time
def generate_category_spend_sql(categories_df):
    sql_parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        sql_parts.append(f"""
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -2)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag1m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -3)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag2m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -4)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -4)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_6m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -4)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.count_charge, 0) else 0
            end) as cnt_cat_{suffix}_3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.count_charge, 0) else 0
            end) as cnt_cat_{suffix}_6m
        """)

    return ",\n".join(sql_parts)

In [ ]:
%%time
category_spend_sql = generate_category_spend_sql(categories_df)

drop = f"""
drop table if exists {cb_category_spend_wide}
"""

create = f"""
create table {cb_category_spend_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    -- общие траты клиента, lag_1m = последний закрытый месяц
    sum(case
            when t.month_date = add_months(b.forecast_month, -2)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_1m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -3)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_2m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -4)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_3m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -5)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_4m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -6)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_5m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -7)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_6m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -4)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_3m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_6m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -2)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_lag_1m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -4)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_3m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_6m,
    -- wide-фичи по категориям
    {category_spend_sql}
from {cb_base} b
left join {cb_charges_compact} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_category_spend_wide, preview_table=cb_category_spend_wide)

### 4. Предложенные категории

In [ ]:
%%time
def generate_offered_percent_sql(categories_df):
    sql_parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        sql_parts.append(f"""
        max(case
                when o.category_id = '{category_id}'
                then cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)
                else 0
            end) as offered_percent_cat_{suffix}
        """)

    return ",\n".join(sql_parts)

offered_percent_sql = generate_offered_percent_sql(categories_df)

drop = f"""
drop table if exists {cb_offered_percent_wide}
"""

create = f"""
create table {cb_offered_percent_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id,
    count(distinct o.category_id) as num_offered_categories,
    max(cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)) as max_offered_percent,
    avg(cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)) as avg_offered_percent,
    {offered_percent_sql}
from {cb_base_scen} b
left join {send_to_dbo} o
    on b.cdi_id = cast(o.cdi_id as bigint)
   and b.forecast_month = cast(o.period_start_date as timestamp)
   and b.offer_scenario_id = o.offer_scenario_id
group by
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id
"""

run_sql(conn, [drop, create], compute_stats_table=cb_offered_percent_wide, preview_table=cb_offered_percent_wide)

### 5. История подключенных категорий

In [ ]:
%%time
def generate_selected_history_sql(categories_df):
    parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        parts.append(f"""
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
                then 1 else 0
            end) as was_selected_cat_{suffix}_lag1m,
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then 1 else 0
            end) as was_selected_cat_{suffix}_3m,
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then 1 else 0
            end) as was_selected_cat_{suffix}_6m,
        avg(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
            end) as avg_selected_percent_cat_{suffix}_6m
        """)

    return ",\n".join(parts)

In [ ]:
%%time
selected_history_sql = generate_selected_history_sql(categories_df)

drop = f"""
drop table if exists {cb_selected_history_wide}
"""

create = f"""
create table {cb_selected_history_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    case
        when count(distinct case
            when trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
            then s.category_id
        end) > 0
        then 1 else 0
    end as selected_any_lag_1m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
        then s.category_id
    end) as cnt_selected_categories_lag_1m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then s.category_id
    end) as cnt_unique_selected_categories_3m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then s.category_id
    end) as cnt_unique_selected_categories_6m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then concat(cast(trunc(cast(s.end_date as timestamp), 'MM') as string), '', cast(s.category_id as string))
    end) as cnt_selected_category_months_3m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then concat(cast(trunc(cast(s.end_date as timestamp), 'MM') as string), '', cast(s.category_id as string))
    end) as cnt_selected_category_months_6m,
    max(case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
    end) as max_selected_percent_6m,
    avg(case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
    end) as avg_selected_percent_6m,
    {selected_history_sql}
from {cb_base} b
left join {LOYALTY_CLIENT_CATEGORIES} s
    on b.cdi_id = cast(s.party_id as bigint)
   and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
   and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(
    conn,
    [drop, create],
    compute_stats_table=cb_selected_history_wide,
    preview_table=cb_selected_history_wide
)

### 6. ACRM targets

In [ ]:
%%time
drop = f"""
drop table if exists {cb_acrm_monthly}
"""

create = f"""
create table {cb_acrm_monthly} stored as parquet as
select
    cast(b.party_id as bigint) as cdi_id,
    cast(trunc(cast(a.transaction_date as date), 'MM') as timestamp) as month_date,
    least(sum(coalesce(a.coins, 0)), 5000) as cashback_month,
    sum(coalesce(a.transaction_amount, 0)) as pos_turnover_month,
    count(*) as pos_txn_cnt_month
from acrm.o_calculations a
join acrm.o_client b
    on a.client_id = b.id
join (
    select distinct cdi_id
    from {cb_base}
) base
    on cast(b.party_id as bigint) = base.cdi_id
where cast(a.transaction_date as date) >= add_months(cast('{start_month}' as timestamp), -7)
  and cast(a.transaction_date as date) < add_months(cast('{end_month}' as timestamp), 1)
group by
    cast(b.party_id as bigint),
    cast(trunc(cast(a.transaction_date as date), 'MM') as timestamp)
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_monthly, preview_table=cb_acrm_monthly)

In [ ]:
%%time
drop = f"""
drop table if exists {cb_acrm_targets}
"""

create = f"""
create table {cb_acrm_targets} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,
    coalesce(t.cashback_month, 0) as target_cashback_month,
    coalesce(t.pos_turnover_month, 0) as target_pos_turnover_month,
    coalesce(t.pos_txn_cnt_month, 0) as target_pos_txn_cnt_month
from {cb_base} b
left join {cb_acrm_monthly} t
    on b.cdi_id = t.cdi_id
   and b.forecast_month = t.month_date
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_targets, preview_table=cb_acrm_targets)

### 7. ACRM lags

In [ ]:
%%time
drop = f"""
drop table if exists {cb_acrm_lags}
"""

create = f"""
create table {cb_acrm_lags} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    sum(case when t.month_date = add_months(b.forecast_month, -2)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_1m,
    sum(case when t.month_date = add_months(b.forecast_month, -3)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_2m,
    sum(case when t.month_date = add_months(b.forecast_month, -4)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_6m,
    avg(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) end) as cashback_avg_3m,
    stddev(case when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.cashback_month, 0) end) as cashback_std_6m,
    sum(case when t.month_date = add_months(b.forecast_month, -2)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_1m,
    sum(case when t.month_date = add_months(b.forecast_month, -3)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_2m,
    sum(case when t.month_date = add_months(b.forecast_month, -4)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_6m,
    avg(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) end) as pos_turnover_avg_3m,
    stddev(case when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.pos_turnover_month, 0) end) as pos_turnover_std_6m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_txn_cnt_month, 0) else 0 end) as pos_txn_cnt_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_txn_cnt_month, 0) else 0 end) as pos_txn_cnt_6m
from {cb_base} b
left join {cb_acrm_monthly} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_lags, preview_table=cb_acrm_lags)

### 8. Activity feats

In [ ]:
%%time
drop = f"""
drop table if exists {cb_activity_features}
"""

create = f"""
create table {cb_activity_features} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    count(distinct case
        when t.month_date = add_months(b.forecast_month, -2)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_1m,
    count(distinct case
        when t.month_date >= add_months(b.forecast_month, -4)
         and t.month_date < add_months(b.forecast_month, -1)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_3m,
    count(distinct case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_6m,
    stddev(case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
        then coalesce(t.charge, 0)
    end) as category_spend_std_6m,
    avg(case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
        then coalesce(t.charge, 0)
    end) as category_spend_avg_6m,
    case
        when avg(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0)
        end) > 0
        then
            stddev(case
                when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0)
            end)
            /
            avg(case
                when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0)
            end)
        else 0
    end as category_spend_cv_6m
from {cb_base} b
left join {cb_charges_compact} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_activity_features, preview_table=cb_activity_features)

### 9. Потенциальный кешбэк

In [ ]:
%%time
drop = f"""
drop table if exists {cb_potential_features}
"""

create = f"""
create table {cb_potential_features} stored as parquet as
with spend_3m_by_category as (
    select
        b.cdi_id,
        b.forecast_month,
        t.title_id as category_id,
        sum(coalesce(t.charge, 0)) as spend_3m,
        sum(coalesce(t.count_charge, 0)) as cnt_txn_3m
    from {cb_base} b
    join {cb_charges_compact} t
        on b.cdi_id = t.cdi_id
       and t.month_date >= add_months(b.forecast_month, -4)
       and t.month_date < add_months(b.forecast_month, -1)
    group by
        b.cdi_id,
        b.forecast_month,
        t.title_id
),
offered as (
    select
        cast(cdi_id as bigint) as cdi_id,
        cast(period_start_date as timestamp) as forecast_month,
        offer_scenario_id,
        category_id,
        max(cast(regexp_replace(cast(category_percent as string), ',', '.') as double)) as offered_percent
    from {send_to_dbo}
    group by
        cast(cdi_id as bigint),
        cast(period_start_date as timestamp),
        offer_scenario_id,
        category_id
)
select
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id,
    sum(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_sum_offered,
    max(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_max_offered,
    avg(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_avg_offered,
    count(distinct case
        when coalesce(s.spend_3m, 0) > 0
        then o.category_id
    end) as cnt_offered_categories_with_spend_3m,
    sum(coalesce(s.spend_3m, 0)) as offered_categories_spend_3m,
    sum(coalesce(s.cnt_txn_3m, 0)) as offered_categories_cnt_txn_3m
from {cb_base_scen} b
left join offered o
    on b.cdi_id = o.cdi_id
   and b.forecast_month = o.forecast_month
   and b.offer_scenario_id = o.offer_scenario_id
left join spend_3m_by_category s
    on o.cdi_id = s.cdi_id
   and o.forecast_month = s.forecast_month
   and o.category_id = s.category_id
group by
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id
"""

run_sql(conn, [drop, create], compute_stats_table=cb_potential_features, preview_table=cb_potential_features)

### 10. Share-фичи

In [ ]:
%%time
categories_df

In [ ]:
%%time
def generate_share_sql(categories_df):
    parts = []

    for _, row in categories_df.iterrows():
        suffix = row["suffix"]

        parts.append(f"""
        case
            when coalesce(total_spend_3m, 0) > 0
            then coalesce(spend_cat_{suffix}_3m, 0) / total_spend_3m
            else 0
        end as share_spend_cat_{suffix}_3m
        """)

    return ",\n".join(parts)


share_sql = generate_share_sql(categories_df)

In [ ]:
%%time
drop = f"""
drop table if exists {cb_share_features}
"""

create = f"""
create table {cb_share_features} stored as parquet as
select
    cdi_id,
    forecast_month,
    {share_sql}
from {cb_category_spend_wide}
"""

run_sql(conn, [drop, create], compute_stats_table=cb_share_features, preview_table=cb_share_features)

In [ ]:
%%time
share_cols_sql = ",\n    ".join([
    f"coalesce(sh.share_spend_cat_{row['suffix']}_3m, 0) as sharespend_cat_{row['suffix']}_3m"
    for _, row in categories_df.iterrows()
])

selected_history_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]

    selected_history_cols.extend([
        f"coalesce(sel.was_selected_cat_{suffix}_lag1m, 0) as was_selected_cat_{suffix}_lag1m",
        f"coalesce(sel.was_selected_cat_{suffix}_3m, 0) as wasselected_cat_{suffix}_3m",
        f"coalesce(sel.was_selected_cat_{suffix}_6m, 0) as wasselected_cat_{suffix}_6m",
        f"coalesce(sel.avg_selected_percent_cat_{suffix}_6m, 0) as avgselected_percent_cat_{suffix}_6m",
    ])

selected_history_cols_sql = ",\n    ".join(selected_history_cols)

In [ ]:
%%time
spend_wide_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]
    spend_wide_cols.extend([
        f"coalesce(sp.spend_cat_{suffix}_lag1m, 0) as spend_cat_{suffix}_lag1m",
        f"coalesce(sp.spend_cat_{suffix}_lag2m, 0) as spend_cat_{suffix}_lag2m",
        f"coalesce(sp.spend_cat_{suffix}_lag3m, 0) as spend_cat_{suffix}_lag3m",
        f"coalesce(sp.spend_cat_{suffix}_3m, 0) as spendcat_{suffix}_3m",
        f"coalesce(sp.spend_cat_{suffix}_6m, 0) as spendcat_{suffix}_6m",
        f"coalesce(sp.cnt_cat_{suffix}_3m, 0) as cntcat_{suffix}_3m",
        f"coalesce(sp.cnt_cat_{suffix}_6m, 0) as cntcat_{suffix}_6m",
    ])

spend_wide_cols_sql = ",\n    ".join(spend_wide_cols)


offered_wide_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]
    offered_wide_cols.append(
        f"coalesce(off.offered_percent_cat_{suffix}, 0) as offered_percent_cat_{suffix}"
    )

offered_wide_cols_sql = ",\n    ".join(offered_wide_cols)

### 11. Финальная витрина

In [ ]:
%%time
drop = f"""
drop table if exists {cb_dataset}
"""

create = f"""
create table {cb_dataset} stored as parquet as
with loyalty_connected as (
    select
        cast(party_id as bigint) as cdi_id,
        trunc(cast(end_date as timestamp), 'MM') as forecast_month
    from {LOYALTY_CLIENT_CATEGORIES}
    group by
        cast(party_id as bigint),
        trunc(cast(end_date as timestamp), 'MM')
)
select
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id,
    -- targets
    coalesce(trg.target_cashback_month, 0) as target_cashback_month,
    coalesce(trg.target_pos_turnover_month, 0) as target_pos_turnover_month,
    coalesce(trg.target_pos_txn_cnt_month, 0) as target_pos_txn_cnt_month,
    case
        when lc.cdi_id is not null then 1
        else 0
    end as target_loyalty_connected_flg,
    -- общие траты клиента
    coalesce(sp.total_spend_lag_1m, 0) as total_spend_lag_1m,
    coalesce(sp.total_spend_lag_2m, 0) as total_spend_lag_2m,
    coalesce(sp.total_spend_lag_3m, 0) as total_spend_lag_3m,
    coalesce(sp.total_spend_lag_4m, 0) as total_spend_lag_4m,
    coalesce(sp.total_spend_lag_5m, 0) as total_spend_lag_5m,
    coalesce(sp.total_spend_lag_6m, 0) as total_spend_lag_6m,
    coalesce(sp.total_spend_3m, 0) as total_spend_3m,
    coalesce(sp.total_spend_6m, 0) as total_spend_6m,
    coalesce(sp.total_cnt_txn_lag_1m, 0) as total_cnt_txn_lag_1m,
    coalesce(sp.total_cnt_txn_3m, 0) as total_cnt_txn_3m,
    coalesce(sp.total_cnt_txn_6m, 0) as total_cnt_txn_6m,
    -- wide-траты по категориям
    {spend_wide_cols_sql},
    -- предложенные категории на forecast_month
    coalesce(off.num_offered_categories, 0) as num_offered_categories,
    coalesce(off.max_offered_percent, 0) as max_offered_percent,
    coalesce(off.avg_offered_percent, 0) as avg_offered_percent,
    {offered_wide_cols_sql},
    -- история выбранных категорий
    coalesce(sel.selected_any_lag_1m, 0) as selected_any_lag_1m,
    coalesce(sel.cnt_selected_categories_lag_1m, 0) as cnt_selected_categories_lag_1m,
    coalesce(sel.cnt_unique_selected_categories_3m, 0) as cnt_unique_selected_categories_3m,
    coalesce(sel.cnt_unique_selected_categories_6m, 0) as cnt_unique_selected_categories_6m,
    coalesce(sel.max_selected_percent_6m, 0) as max_selected_percent_6m,
    coalesce(sel.avg_selected_percent_6m, 0) as avg_selected_percent_6m,
    {selected_history_cols_sql},
    -- ACRM lags
    coalesce(al.cashback_lag_1m, 0) as cashback_lag_1m,
    coalesce(al.cashback_lag_2m, 0) as cashback_lag_2m,
    coalesce(al.cashback_lag_3m, 0) as cashback_lag_3m,
    coalesce(al.cashback_3m, 0) as cashback_3m,
    coalesce(al.cashback_6m, 0) as cashback_6m,
    coalesce(al.cashback_avg_3m, 0) as cashback_avg_3m,
    coalesce(al.cashback_std_6m, 0) as cashback_std_6m,
    coalesce(al.pos_turnover_lag_1m, 0) as pos_turnover_lag_1m,
    coalesce(al.pos_turnover_lag_2m, 0) as pos_turnover_lag_2m,
    coalesce(al.pos_turnover_lag_3m, 0) as pos_turnover_lag_3m,
    coalesce(al.pos_turnover_3m, 0) as pos_turnover_3m,
    coalesce(al.pos_turnover_6m, 0) as pos_turnover_6m,
    coalesce(al.pos_turnover_avg_3m, 0) as pos_turnover_avg_3m,
    coalesce(al.pos_turnover_std_6m, 0) as pos_turnover_std_6m,
    coalesce(al.pos_txn_cnt_3m, 0) as pos_txn_cnt_3m,
    coalesce(al.pos_txn_cnt_6m, 0) as pos_txn_cnt_6m,
    -- активность
    coalesce(act.cnt_categories_with_spend_1m, 0) as cnt_categories_with_spend_1m,
    coalesce(act.cnt_categories_with_spend_3m, 0) as cnt_categories_with_spend_3m,
    coalesce(act.cnt_categories_with_spend_6m, 0) as cnt_categories_with_spend_6m,
    coalesce(act.category_spend_std_6m, 0) as category_spend_std_6m,
    coalesce(act.category_spend_avg_6m, 0) as category_spend_avg_6m,
    coalesce(act.category_spend_cv_6m, 0) as category_spend_cv_6m,
    -- potential cashback
    coalesce(p.potential_cashback_sum_offered, 0) as potential_cashback_sum_offered,
    coalesce(p.potential_cashback_max_offered, 0) as potential_cashback_max_offered,
    coalesce(p.potential_cashback_avg_offered, 0) as potential_cashback_avg_offered,
    coalesce(p.cnt_offered_categories_with_spend_3m, 0) as cnt_offered_categories_with_spend_3m,
    coalesce(p.offered_categories_spend_3m, 0) as offered_categories_spend_3m,
    coalesce(p.offered_categories_cnt_txn_3m, 0) as offered_categories_cnt_txn_3m,
    -- share categories
    {share_cols_sql},
    -- calendar
    year(b.forecast_month) as forecast_year,
    month(b.forecast_month) as forecast_month_num
from {cb_base_scen} b
left join {cb_acrm_targets} trg
    on b.cdi_id = trg.cdi_id
   and b.forecast_month = trg.forecast_month
left join {cb_category_spend_wide} sp
    on b.cdi_id = sp.cdi_id
   and b.forecast_month = sp.forecast_month
left join {cb_offered_percent_wide} off
    on b.cdi_id = off.cdi_id
   and b.forecast_month = off.forecast_month
   and b.offer_scenario_id = off.offer_scenario_id
left join {cb_selected_history_wide} sel
    on b.cdi_id = sel.cdi_id
   and b.forecast_month = sel.forecast_month
left join {cb_acrm_lags} al
    on b.cdi_id = al.cdi_id
   and b.forecast_month = al.forecast_month
left join {cb_activity_features} act
    on b.cdi_id = act.cdi_id
   and b.forecast_month = act.forecast_month
left join {cb_potential_features} p
    on b.cdi_id = p.cdi_id
   and b.forecast_month = p.forecast_month
   and b.offer_scenario_id = p.offer_scenario_id
left join {cb_share_features} sh
    on b.cdi_id = sh.cdi_id
   and b.forecast_month = sh.forecast_month
left join loyalty_connected lc
    on b.cdi_id = lc.cdi_id
   and b.forecast_month = lc.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_dataset, preview_table=cb_dataset)

In [ ]:
%%time
cb_dataset

### 12. JOIN фичестора

In [ ]:
%%time
with conn:
    d1 = conn.fetch(f'''select * from {cb_dataset} where forecast_month = '{TARGET_MONTH}' limit 10''')
    df_all = conn.fetch(f'''select count(*) from {cb_dataset} limit 5''')
d1

In [ ]:
%%time
df_all

In [ ]:
MODULAR_FEATURE_STORE = "sandbox_crft.feature_store_inc_modular"
with conn:
    d3 = conn.fetch(f"SELECT * FROM {MODULAR_FEATURE_STORE} LIMIT 1")
d3

In [ ]:
# Отбираем только признаки modular FS, не перезаписывая базовые признаки.
base_cols = list(d1.columns)
modular_cols = [c for c in d3.columns if c not in set(base_cols)
                and c not in {"cdi_id", "report_date"}
                and not c.startswith("loyal__")
                and not c.endswith(("_date", "_dttm", "_cust_id"))]
print("base:", len(base_cols), "modular:", len(modular_cols))

In [ ]:
# Для modular FS используем явную квалификацию ld / cs, чтобы исключить неоднозначность имён.

In [ ]:
select_clause = ",\n        ".join(
    [f"ld.`{name}` AS `{name}`" for name in base_cols] +
    [f"cs.`{name}` AS `{name}`" for name in modular_cols]
)

In [ ]:
conn_hive = connect(to="IMPALA")  # текущая Kerberos-сессия; без пароля в ноутбуке

In [ ]:
%%time
DATASET_WITH_FEATURES = f'sandbox_crft.mae_nbc_dataset_with_features_{postfix}'

In [ ]:
%%time
with conn:
    conn.execute(f'''drop table if exists {DATASET_WITH_FEATURES}''')

In [ ]:
with conn_hive:
    conn_hive.execute(f'''CREATE TABLE {DATASET_WITH_FEATURES}
        STORED AS PARQUET TBLPROPERTIES('transactional'='false') AS
        SELECT {select_clause}
        FROM {cb_dataset} ld
        LEFT JOIN {MODULAR_FEATURE_STORE} cs
          ON CAST(cs.cdi_id AS BIGINT) = CAST(ld.cdi_id AS BIGINT)
         AND YEAR(CAST(cs.report_date AS DATE))*100 + MONTH(CAST(cs.report_date AS DATE))
             = YEAR(ADD_MONTHS(CAST(ld.forecast_month AS DATE),-2))*100
               + MONTH(ADD_MONTHS(CAST(ld.forecast_month AS DATE),-2))''')

In [ ]:
%%time
MUST_HAVE = get_feature_list('data_fm_must_have_features')
COLS_BIN_LOYAL = get_feature_list('train_pipeline_loyalty_binary_columns')
COLS_CASHBACK = get_feature_list('train_pipeline_cashback_columns')
SELECT_POS_LOYAL_COLS = get_feature_list('data_fm_pos_loyalty_columns')
SELECT_POS_NOLOYAL_COLS = get_feature_list('data_fm_pos_noloyalty_columns')
SELECT_DELTA_LOYAL_COLS = get_feature_list('data_fm_delta_loyalty_columns')
SELECT_DELTA_NOLOYAL_COLS = get_feature_list('data_fm_delta_noloyalty_columns')
SELECT_BINARY_LOYAL_COLS = get_feature_list('data_fm_binary_loyalty_columns')
SELECT_BINARY_NOLOYAL_COLS = get_feature_list('data_fm_binary_noloyalty_columns')

In [ ]:
total_cols_sql = "*"
# Dynamic: полный набор колонок сохраняем в Parquet; модельные колонки читает PyArrow по feature_names_.

In [ ]:
%%time
cb_infer_data = f"sandbox_crft.cb_infer_dataset_{postfix}"

drop = f"""
    drop table if exists {cb_infer_data}
"""

create = f"""
    create table {cb_infer_data} stored as parquet as
    select
        {total_cols_sql}
    from {DATASET_WITH_FEATURES} d
"""

with conn_hive:
    for query in [drop, create]:
        conn_hive.execute(query)

In [ ]:
# Копируем строго созданную выше витрину. Никаких фиксированных июльских/ноябрьских имён.
import shutil, subprocess
LOCAL_INFER_PARENT = PROJECT_ROOT / "data" / "datasets" / "inference"
LOCAL_INFER_PARENT.mkdir(parents=True, exist_ok=True)
INFER_TABLE_NAME = cb_infer_data.split(".")[-1]
LOCAL_INFER_DATASET = LOCAL_INFER_PARENT / INFER_TABLE_NAME
HDFS_INFER_DATASET = f"/warehouse/tablespace/managed/hive/sandbox_crft.db/{INFER_TABLE_NAME}"
if LOCAL_INFER_DATASET.exists():
    shutil.rmtree(LOCAL_INFER_DATASET)
subprocess.run(["hdfs", "dfs", "-copyToLocal", HDFS_INFER_DATASET, str(LOCAL_INFER_PARENT)], check=True)
if not LOCAL_INFER_DATASET.is_dir():
    raise FileNotFoundError(LOCAL_INFER_DATASET)
print("Parquet:", LOCAL_INFER_DATASET)


In [ ]:
%%time
cb_infer_data

In [ ]:
%%time
!mkdir -p ./data/datasets/inference

In [ ]:
# Скачивание Parquet выполняется единым блоком ниже, по имени только что созданной таблицы.

In [ ]:
# Скачивание Parquet выполняется единым блоком ниже, по имени только что созданной таблицы.

In [ ]:
1 

## Скоринг

In [ ]:
%%time
# Настройки inference вынесены в configs/inference_config.json
from nbc_project.inference_helpers import (
    load_inference_config,
    build_inference_table_names,
    run_sql,
    category_suffix,
    add_category_suffix,
    generate_category_spend_sql,
    generate_offered_percent_sql,
    generate_selected_history_sql,
    generate_share_sql,
    generate_send_to_dbo_scenario_mart_sql,
    add_offer_scenario_filter,
    read_partitioned_parquet_folder,
    optimize_dataframe_types,
    prepare_inference_frame,
    load_catboost_models,
    load_optional_calibrators,
    model_output_columns,
    predict_with_models,
    infer_required_input_columns,
    score_inference_parquet_in_batches,
    upload_folder_to_hdfs,
)

INFERENCE_CONFIG = load_inference_config(PROJECT_ROOT)
INFERENCE_CONFIG["preprocessing_mode"] = "training_aligned"
INFERENCE_CONFIG["strict_features"] = True


In [ ]:
%%time
# Режимы inference. Для больших данных RUN_BATCH_INFERENCE=True.
RUN_BATCH_INFERENCE = True
RUN_FULL_INFERENCE = False

INFERENCE_DATASET_PATH = str(LOCAL_INFER_DATASET)
BATCH_INFERENCE_OUTPUT_DIR = PROJECT_ROOT / "scores" / postfix
BATCH_SIZE = 500000
BATCH_FILE_FORMAT = 'csv'  # csv удобно для текущей загрузки в HDFS; можно поставить 'parquet'

In [ ]:
%%time
# Все 8 моделей обязательны; загрузка старых июльских/сентябрьских запрещена.
# Модели соответствующей ветки: пути определены в configs/inference_config.json.
# Ноябрьские модели for_november не используются автоматически.
model_paths = INFERENCE_CONFIG['model_paths']
for _key, _spec in model_paths.items():
    _file = PROJECT_ROOT / _spec['path']
    if not _file.is_file():
        raise FileNotFoundError(f'Не найдена октябрьская модель {_key}: {_file}. Сначала выполните train 2.1–2.4')
    if _spec.get('calibrator_path') and not (PROJECT_ROOT / _spec['calibrator_path']).is_file():
        raise FileNotFoundError(f'Нет калибратора {_key}: {_spec["calibrator_path"]}')
models = load_catboost_models(model_paths, project_root=PROJECT_ROOT)
calibrators = load_optional_calibrators(model_paths, project_root=PROJECT_ROOT)
output_columns = model_output_columns(model_paths)

models.keys()

In [ ]:
%%time
inference_input_columns = infer_required_input_columns(
    models=models,
    config=INFERENCE_CONFIG,
    id_columns=['cdi_id', 'client_id', 'forecast_month', 'period_start_date', 'offer_scenario_id'],
)

print(f'Колонок для чтения: {len(inference_input_columns)}')
inference_input_columns[:20]

In [ ]:
# Важно: удаляем прошлые scores_part только в персональной директории запуска.
import shutil
if RUN_BATCH_INFERENCE and BATCH_INFERENCE_OUTPUT_DIR.exists():
    shutil.rmtree(BATCH_INFERENCE_OUTPUT_DIR)
if RUN_BATCH_INFERENCE:
    score_part_paths = score_inference_parquet_in_batches(
        input_path=INFERENCE_DATASET_PATH,
        output_dir=BATCH_INFERENCE_OUTPUT_DIR,
        models=models,
        inference_config=INFERENCE_CONFIG,
        output_columns=output_columns,
        calibrators=calibrators,
        columns=inference_input_columns,
        batch_size=BATCH_SIZE,
        score_columns=INFERENCE_CONFIG['score_columns'],
        offer_batch_func=None,
        file_format=BATCH_FILE_FORMAT,
    )

    print(f'Готово part-файлов: {len(score_part_paths)}')

In [ ]:
%%time
import pandas as pd
# Лёгкая проверка результата без чтения всех part-файлов в память.
if RUN_BATCH_INFERENCE and BATCH_FILE_FORMAT == 'csv':
    first_part = sorted(BATCH_INFERENCE_OUTPUT_DIR.glob('scores_part_*.csv'))[0]
    display(pd.read_csv(first_part, nrows=10))
elif RUN_BATCH_INFERENCE and BATCH_FILE_FORMAT == 'parquet':
    first_part = sorted(BATCH_INFERENCE_OUTPUT_DIR.glob('scores_part_*.parquet'))[0]
    display(pd.read_parquet(first_part).head())

In [ ]:
%%time
# Старый full-inference оставлен только для маленьких выборок/debug.
# На больших данных RUN_FULL_INFERENCE должен быть False, иначе kernel может упасть.
if RUN_FULL_INFERENCE:
    df_infer = read_partitioned_parquet_folder(INFERENCE_DATASET_PATH)
    df_infer = prepare_inference_frame(
        df_infer,
        pos_values=INFERENCE_CONFIG['pos_values'],
        season_base_cols=INFERENCE_CONFIG['season_base_cols'],
        cat_features=INFERENCE_CONFIG['cat_features'],
        numeric_cols=INFERENCE_CONFIG['numeric_string_cols'],
    )
    df_infer = predict_with_models(
        df_infer,
        models=models,
        output_columns=output_columns,
        calibrators=calibrators,
    )
    if {'cashback_pred', 'loyal_pred'}.issubset(df_infer.columns):
        df_infer['cashback_expect'] = df_infer['cashback_pred'] * df_infer['loyal_pred']

    score_columns = [col for col in INFERENCE_CONFIG['score_columns'] if col in df_infer.columns]
    df_infer[score_columns].to_csv(BATCH_INFERENCE_OUTPUT_DIR / "debug_full_inference.csv", index=False)
    display(df_infer[score_columns].head())


In [ ]:
HDFS_SCORE_PATH = "/warehouse/tablespace/external/hive/stg_datamarts.db/temp_uploads/dec_active_base_dynamic_2026"
# Создаём директорию отдельно для сценария и ветки.
import subprocess
# Не смешивать прошлые score parts со свежими: если файлы уже есть, остановиться.
_check = subprocess.run(['hdfs','dfs','-ls',HDFS_SCORE_PATH],capture_output=True,text=True)
if _check.returncode == 0 and any(x.lstrip().startswith(('-', 'd')) and 'scores_part_' in x for x in _check.stdout.splitlines()):
    raise FileExistsError(f'HDFS {HDFS_SCORE_PATH} уже содержит scores_part_*: проверьте старые файлы перед перезапуском')
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", HDFS_SCORE_PATH], check=True)
# Загрузка файлов в HDFS вынесена в upload_folder_to_hdfs().
def loading_at_file(source_folder):
    return upload_folder_to_hdfs(
        source_folder=source_folder,
        hdfs_path=HDFS_SCORE_PATH,
        namenode_url=INFERENCE_CONFIG['hdfs']['namenode_url'],
    )

In [ ]:
%%time
import os
SAVE_DF_PATH = os.path.join('/home/jovyan/Artem/NBC/project_nbc_scenarios/export') ## папка для сохранения 

In [ ]:
!pip install hdfs

In [ ]:
%%time
from rail_connectors import connect
import rail_connectors

conn_hive = connect(to='DATALAKE')

In [ ]:
%%time
conn_impala = connect(to='IMPALA')

In [ ]:
BATCH_INFERENCE_OUTPUT_DIR

In [ ]:
# Не загружаем файлы двух веток в один общий каталог.
HDFS_SCORE_PATH = "/warehouse/tablespace/external/hive/stg_datamarts.db/temp_uploads/dec_active_base_dynamic_2026"
loading_at_file(BATCH_INFERENCE_OUTPUT_DIR)

In [ ]:
%%time
with conn_hive:
    for query in [
        f"drop table if exists sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026",
        f"""
            CREATE EXTERNAL TABLE sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026 (
                `cdi_id` bigint,
                `offer_scenario_id` string,
                `pos_loyal_pred` double,
                `pos_noloyal_pred` double,
                `binary_loyal_pred` double,
                `binary_noloyal_pred` double,
                `delta_loyal_pred` double,
                `delta_noloyal_pred` double,
                `cashback_pred` double,
                `loyal_pred` double,
                `cashback_expect` double,
                `pos_expect` double,
                `batch_num` integer
              )
              ROW FORMAT SERDE 
              'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe' 
            WITH SERDEPROPERTIES ( 
              'field.delim'=',',
              'serialization.null.format'='') 
            STORED AS INPUTFORMAT 
              'org.apache.hadoop.mapred.TextInputFormat' 
            OUTPUTFORMAT 
              'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat'
            LOCATION
              '{HDFS_SCORE_PATH}'
            TBLPROPERTIES (
              'bucketing_version'='2', 
              'numFilesErasureCoded'='0', 
              'skip.header.line.count'='1', 
              'transient_lastDdlTime'='1706202405')
        """
    ]:
        conn_hive.execute(query)
print('ХАЙВ')

In [ ]:
%%time
with conn_impala:
    for query in [
        f"drop table if exists sandbox_crft.mae_nbc_optimizer_scores_dec_active_base_dynamic_2026",
        f"invalidate metadata sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026",
        f"""
        create table sandbox_crft.mae_nbc_optimizer_scores_dec_active_base_dynamic_2026 TBLPROPERTIES ('transactional'='false') as
    	select *
    	from sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026""",
        f"drop table if exists sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026"
    ]:
        conn_impala.execute(query)

In [ ]:
%%time
# !rm -rf scores/october/infer_oct_pos_parts